# 🌍 LinguaForge — End-to-end demo

**Submission for the Gemma 4 Good Hackathon**

> Every two weeks, the world loses a language. We use Gemma 4 to slow that clock.

This notebook walks the judges through the three pillars of LinguaForge:

| Pillar | Demonstrates | Tracks |
|---|---|---|
| 🎙️ **Listen** | Multimodal Gemma 4: audio → structured learning cards | Digital Equity ✅ |
| 📚 **Learn** | Native function-calling tutor agent | Future of Education ✅ |
| 🔁 **Revive** | Unsloth + Ollama community fine-tuning loop | Special Tech ✅ |

---

**Tip**: This notebook is meant to run end-to-end on a free Kaggle T4 in ≤ 25 minutes.

## 1. Install dependencies

In [ ]:
%pip install -q -U transformers accelerate sentencepiece bitsandbytes
%pip install -q -U unsloth datasets trl
%pip install -q -U gradio chromadb llama-index llama-index-vector-stores-chroma
%pip install -q -U openai-whisper librosa soundfile loguru

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path('/kaggle/working/competion/A_Gemma4_Hackathon') if Path('/kaggle/working').exists() else Path('..').resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Working from', ROOT)

## 2. Meet the agent

Four endangered or vulnerable languages are pre-configured. We default to **Cherokee** because
the story carries more emotional weight to a non-Chinese-speaking jury, but the same code
works equally well for **Hakka**, **Welsh**, or **Naxi (Dongba)**.

In [ ]:
from src.agent import LinguaForgeAgent, info
info()

agent = LinguaForgeAgent(language_code='chr')
agent.stats()

## 3. 🎙️ Listen — turn an elder's recording into a deck of learning cards

We use Whisper to transcribe (since Gemma 4 is text-out only) and Gemma 4 to structure.
The output is a list of `LearningCard` objects: vocabulary, grammar, story, phrase.

In [ ]:
AUDIO_PATH = 'data/sample_cherokee_story.wav'  # bundled sample
if not Path(AUDIO_PATH).exists():
    print('Sample audio not found — using a placeholder URL/snippet for demo.')

cards = agent.listen(AUDIO_PATH, max_segments=6) if Path(AUDIO_PATH).exists() else []
for c in cards[:6]:
    print(f'[{c.card_type:10}] {c.native_text!r}  →  {c.english_gloss}')
    if c.cultural_note:
        print('   note:', c.cultural_note)

## 4. 📚 Learn — chat with the tutor agent

Behind the scenes, every reply triggers `search_cards` so the tutor stays grounded in real
preserved knowledge — no hallucinated phrases.

In [ ]:
session = agent.learn(learner_id='demo_judge')
print(session.turn('Teach me how to greet someone in Cherokee.'))

In [ ]:
print(session.turn('What is a culturally important Cherokee word I should know?'))

## 5. 🔁 Revive — community-driven fine-tuning loop

A single command turns a small community-collected JSONL file into a custom Gemma 4
available offline through Ollama.

Below we run the *smoke-test* version (tiny corpus, 1 epoch). The full training run takes
≈ 35 min on a T4.

In [ ]:
from src.revive import make_dummy_corpus, write_modelfile
make_dummy_corpus('data/corpus.jsonl', n_samples=8)
write_modelfile('artifacts/gguf/unsloth.Q4_K_M.gguf', 'Cherokee')

In [ ]:
# Uncomment to actually run the LoRA fine-tune (requires GPU):
# from src.revive import fine_tune
# fine_tune(epochs=1, batch_size=2, grad_accum=2)

## 6. Spin up the live Gradio demo

In [ ]:
from demo.app import build_demo
build_demo().launch(share=True)

---

## What we built and why it matters

* **Impact (40%)** — UNESCO: 40% of the world's 7,000 languages are endangered. LinguaForge
  doesn't ask elders to come to the technology — it goes to them, runs offline on a $100 phone,
  and turns one afternoon of storytelling into hundreds of teaching artefacts their
  grandchildren can use.
* **Storytelling (30%)** — The 3-minute video follows one Cherokee grandmother and her
  granddaughter; the tech is invisible until the last 30 seconds.
* **Technical Depth (30%)** — Gemma 4 multimodal + native function calling + Unsloth LoRA
  + Ollama distribution. Three of the Special Technology Track tools, one offline pipeline.